# 06 Base model A

Predicts the taxi-out time from all 2025 departures, including the unusual ones (artefacts, very long taxi times). It is
the anchor of the corrector in `09`.

In [1]:
OOF_MONTHS = None        # None = an out-of-fold prediction for every month of 2025
MAX_PARALLEL_BLOCKS = 6

## Features

In [2]:
FEATURES = [
    # airport, aircraft and runway
    "ADEP_mvt", "AIRCRAFT_TYPE_mvt", "RUNWAY_mvt", "AIRCRAFT_OPERATOR_flt",
    # NM and schedule times
    "mvt_minus_aobt_sec", "movement_minus_scheduled_offblock_sec", "mvt_minus_lobt_sec", "aobt_minus_eobt_sec",
    "lobt_minus_eobt_sec",
    # traffic, weather and day of week
    "deps_before_30min", "deps_before_30min_same_runway", "deps_after_30min_same_runway", "seconds_since_last_movement",
    "dep_median_delay_30min_sec", "weather_temperature_c", "weather_visibility_km",
    "movement_day_of_week_sin", "movement_day_of_week_cos",
    # ADS-B and stand
    "adsb_movement_to_mvt_sec", "adsb_first_observation_to_mvt_sec", "adsb_movement_to_runway_direct_m",
    "stand_reocc_gap_sec", "stand_reocc_other_type", "stand_prev_arr_gap_sec",
]

## Model

CatBoost regressor, depth 8, learning rate 0.05, up to 20 000 trees with early stopping; deeper trees than the usual
depth 6 gave better out-of-fold results. The four text columns (airport, aircraft type, runway, operator) are passed
as categorical features.

In [3]:
from catboost import CatBoostRegressor

MODEL = CatBoostRegressor(iterations=20000, depth=8, learning_rate=0.05, border_count=4095, loss_function="RMSE",
                          cat_features=["ADEP_mvt", "AIRCRAFT_TYPE_mvt", "RUNWAY_mvt", "AIRCRAFT_OPERATOR_flt"],
                          verbose=False, allow_writing_files=False)

## Data

In [4]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model/runner.py").is_file())
sys.path.insert(0, str(ROOT / "model"))
import runner
from runner import TARGET, TIME

runner.configure_tracking()
DYNAMIC = {name: (runner.FEATURE_DIR / f"{name}.py").read_text() for name in ["median_taxi_out"]}
needed = list(dict.fromkeys(FEATURES + ["STAND_mvt", "MVT_TIME_UTC_mvt", TARGET, TIME, *runner.METADATA]))
data = pd.read_parquet(ROOT / "data/03_features/train_DEP_merged_sorted.parquet", columns=needed)
data = data.loc[pd.to_datetime(data[TIME], utc=True).dt.year.eq(2025)].reset_index(drop=True)
print(f"{len(data):,} departures in 2025, {len(FEATURES)} features + {list(DYNAMIC)}")

2,085,041 departures in 2025, 24 features + ['median_taxi_out']


In [5]:
fit_mask = np.ones(len(data), dtype=bool)   # base A learns from every 2025 departure

## Month blocks

Each month of 2025 is predicted by a model trained on the other eleven months. Early stopping uses the month before
the block (for January the month after it); with that number of trees the model is then refitted on all eleven
months. These out-of-fold predictions are what the corrector learns from.

In [6]:
blocks = runner.month_blocks(OOF_MONTHS)
run, oof, metrics = runner.train_blocks("BaseA_CatBoost", MODEL, data, FEATURES + list(DYNAMIC), DYNAMIC, fit_mask, blocks,
                                        MAX_PARALLEL_BLOCKS, params={"model_role": "BaseA_CatBoost"})
print("Run:", run.info.run_name, run.info.run_id)

Run 291_BaseA_CatBoost: 12 blocks, 6 in parallel with 5 CPUs each


[1/12] m01: RMSE 340.3, trees 1655, 56 min


[2/12] m07: RMSE 351.7, trees 2963, 154 min


[3/12] m03: RMSE 197.6, trees 5931, 198 min


[4/12] m08: RMSE 269.4, trees 2160, 232 min


...

[9/12] m09: RMSE 238.2, trees 9342, 481 min


[10/12] m10: RMSE 209.2, trees 9781, 510 min


[11/12] m12: RMSE 198.8, trees 9747, 554 min


[12/12] m11: RMSE 193.0, trees 14200, 605 min


Run: 291_BaseA_CatBoost 2a97c58055c641429638c1f74fa74751


In [7]:
month = pd.to_datetime(oof[TIME], utc=True).dt.month
error = (oof["prediction"] - oof["target"]) ** 2
table = pd.DataFrame({"rmse": error.groupby(month).mean() ** 0.5,
                      "rmse_le_1h": error[oof["target"] <= 3600].groupby(month[oof["target"] <= 3600]).mean() ** 0.5,
                      "trees": [metrics.get(f"best_iteration_m{m:02d}") for m in sorted(month.unique())]})
table.round(1)

,rmse,rmse_le_1h,trees
SCHED_TIME_UTC_mvt,,,
1,340.3,199.0,1655
2,283.8,187.9,12088
3,197.6,175.5,5931
4,201.3,181.6,9058
5,284.8,181.2,7130
6,303.2,200.6,11234
7,351.7,235.7,2963
8,269.4,217.2,2160
9,238.2,207.6,9342


## Final model

Trained on all allowed 2025 rows with the median number of trees of the blocks, saved in the same MLflow run under
`refit/`. `09_corrector` copies it to `model/final`, where the prediction notebooks (`10`, `11`) read it.

In [8]:
# Visibility is sometimes missing for whole stations in later files; the 2025 airport median fills such gaps at
# prediction time (only airports where 2025 itself has visibility for at least half of the flights).
vis = data.groupby("ADEP_mvt")["weather_visibility_km"]
fill_medians = {"weather_visibility_km": vis.median()[vis.apply(lambda s: s.isna().mean()) < 0.5].dropna().to_dict()}

In [9]:
final, meta = runner.refit(run, MODEL, data, FEATURES + list(DYNAMIC), DYNAMIC, fit_mask, metrics, blocks,
                           extra={"fill_medians": fill_medians})
print(f"Refit on {meta['fit_rows']:,} rows with {meta['trees']} trees -> {run.info.run_name}")

Refit on 2,085,041 rows with 9200 trees -> 291_BaseA_CatBoost
